In [ ]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv


/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11880.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store = Chroma(
    collection_name="Culinary_Knowledge",
    embedding_function=embeddings,
    persist_directory="./Vector_Storage_MasterChef"
)

# 3. Quick Verification
count = vector_store._collection.count()


In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 10, 'fetch_k': 20, 'lambda_mult': 0.5}
)

In [5]:
retriever.invoke("What foods were introduced to Korea through the Mongol invasion of Goryeo?")

[]

In [9]:
print("Collection count:", vector_store._collection.count())

Collection count: 0


In [8]:
vector_store.similarity_search("What foods were introduced to Korea through the Mongol invasion of Goryeo?", k=10)

[]

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 293.32it/s]


In [15]:
from sentence_transformers import CrossEncoder

# Load once, not inside the function
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def generate_response(query):
    docs = stored.similarity_search(query, k=10)
    docs1 = retriever.invoke(query)  # This will print the retrieved documents for debugging
    merged_docs = docs + docs1  # Combine both retrievals for richer context
    seen = set()
    unique_merged_docs = []
    for doc in merged_docs:
        text = doc.page_content.strip()
        if text not in seen:
            seen.add(text)
            unique_merged_docs.append(doc)

    pairs = [(query, doc.page_content) for doc in unique_merged_docs]

    # Step 4: score all candidates
    scores = reranker.predict(pairs)

    # Step 5: sort by score descending and keep top 5
    ranked_docs = sorted(
        zip(unique_merged_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_docs = [doc for doc, score in ranked_docs[:5]]

    # Step 6: build final context from top 5 only
    context = "\n\n".join([doc.page_content for doc in top_docs])
    messages = [
        {"role": "system", "content": "You are a specialist East Asian Master Chef. Use the provided context to answer the user's question accurately and verbosely if there are any explanations. If the answer is not in the context, say you don't know based on the current records. Be concise. make sure you include the source of the information"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    
    # Apply ChatML template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 5. Generate
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.1 # Low temperature for factual accuracy
    )
    
    # Remove the prompt from the output
    response = tokenizer.batch_decode(
        [out[len(in_ids):] for in_ids, out in zip(model_inputs.input_ids, generated_ids)],
        skip_special_tokens=True
    )[0]
    
    return response


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9155.20it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
generate_response("What foods were introduced to Korea through the Mongol invasion of Goryeo?")

'Based on the provided context, the Mongol invasion of Goryeo led to the introduction of several traditional foods that became popular in Korea:\n\n1. Dumplings - Grilled meat dishes like mandu.\n2. Grilled meat dishes - Such as grilled pork and seafood.\n3. Noodle dishes - Often served with seasonings like black pepper.\n4. Seasonings - Including black pepper, which was used extensively in cooking.\n\nThese foods were introduced during the later Goryeo period when the Mongols invaded the region. They played a significant role in shaping Korean cuisine and dietary habits.'

In [3]:
query = "What foods were introduced to Korea through the Mongol invasion of Goryeo?"

In [1]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_chroma import Chroma
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from sentence_transformers import CrossEncoder

vector_store = Chroma(
        collection_name="Culinary_Knowledge",
        embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
        persist_directory="./Vector_Storage_masterChef"
    )

/opt/miniconda3/envs/nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5976.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
similarity_search_results = vector_store.similarity_search(query, k=5)


In [5]:
retriever_results = retriever.invoke(query)

NameError: name 'retriever' is not defined